# Pragmatic naming experiment analysis

Compare how accurately the pragmatic evaluator assigns Naming Understandability Scores with and without the Context Description.

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from pymongo import MongoClient

EXPERIMENTS = {
    "pragmatic-context-gpt5-6-luna": "With context",
    "pragmatic-no-context-gpt5-6-luna": "Without context",
}
COLLECTION_NAME = "pragmatic_naming_eval"
MONGODB_URI = os.getenv(
    "MONGODB_URI",
    "mongodb://127.0.0.1:27017/llm_uml_evaluator?replicaSet=rs0",
)

plt.style.use("seaborn-v0_8-whitegrid")

## Load data

The URI comes from the `MONGODB_URI` environment variable. Start JupyterLab with `uv run --env-file .env --group eval jupyter lab`.

In [ ]:
client = MongoClient(MONGODB_URI, serverSelectionTimeoutMS=5_000)
try:
    collection = client.get_default_database()[COLLECTION_NAME]
    documents = list(
        collection.find(
            {"experiment_name": {"$in": list(EXPERIMENTS)}},
            {
                "experiment_name": 1,
                "sample": 1,
                "mutation": 1,
                "nodes": 1,
            },
        )
    )
finally:
    client.close()

loaded_experiments = {document["experiment_name"] for document in documents}
if missing := set(EXPERIMENTS).difference(loaded_experiments):
    raise ValueError(f"No observations found for: {sorted(missing)}")

pd.Series(
    (document["experiment_name"] for document in documents),
    name="experiment_name",
).value_counts().rename("observations")

## Prepare metrics

Each classification pair contains a node UID and its Naming Understandability Score. A wrong score contributes one false positive and one false negative, so precision, recall, and F1 are identical here. The notebook keeps only F1.

In [ ]:
df = pd.json_normalize(documents).rename(
    columns={
        "nodes.true_positive": "tp",
        "nodes.false_positive": "fp",
        "nodes.false_negative": "fn",
    }
)

required_columns = {
    "experiment_name", "sample", "mutation", "tp", "fp", "fn"
}
if missing := required_columns.difference(df.columns):
    raise ValueError(f"Missing columns: {sorted(missing)}")

df["context"] = df["experiment_name"].map(EXPERIMENTS)
f1_denominator = 2 * df["tp"] + df["fp"] + df["fn"]
df["f1"] = 2 * df["tp"] / f1_denominator.where(
    f1_denominator.ne(0)
)
df["all_nodes_correct"] = df["fp"].eq(0) & df["fn"].eq(0)

df[["experiment_name", "sample", "mutation", "context", "f1", "all_nodes_correct"]].head()

## Check coverage

The table shows the number of repetitions for each `experiment × sample × mutation` combination.

In [ ]:
coverage = (
    df.groupby(["context", "sample", "mutation"])
    .size()
    .unstack(fill_value=0)
)
display(coverage)
display(
    df.groupby(["context", "mutation"])["f1"]
    .agg(["mean", "std", "count"])
    .round(3)
)

## Quality by mutation

Points show mean F1 for each context mode; vertical bars show one standard deviation across observations.

In [ ]:
f1_by_mutation = df.groupby(["context", "mutation"])["f1"].agg(
    ["mean", "std"]
)

fig, ax = plt.subplots(figsize=(10, 5))
for context in EXPERIMENTS.values():
    values = f1_by_mutation.loc[context]
    ax.errorbar(
        values.index,
        values["mean"],
        yerr=values["std"],
        marker="o",
        capsize=4,
        label=context,
    )
ax.set(
    title="Pragmatic naming F1 by mutation",
    xlabel="Mutation",
    ylabel="Score",
    ylim=(0, 1.05),
)
ax.legend()
plt.show()

## All nodes correct rate

An observation counts as correct only when every evaluated node receives the expected Naming Understandability Score.

In [ ]:
all_correct_by_mutation = (
    df.groupby(["mutation", "context"])["all_nodes_correct"]
    .mean()
    .unstack("context")
    .reindex(columns=EXPERIMENTS.values())
)

fig, ax = plt.subplots(figsize=(9, 4))
all_correct_by_mutation.plot.bar(ax=ax, rot=0)
for container in ax.containers:
    ax.bar_label(
        container,
        labels=[f"{value:.0%}" for value in container.datavalues],
        padding=3,
    )
ax.set(
    title="All nodes correct by mutation",
    xlabel="Mutation",
    ylabel="All nodes correct rate",
    ylim=(0, 1.08),
)
plt.show()